In [ ]:
### Sentiment Analayzer

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path().resolve().parent  # because notebook is inside /notebooks
DB_PATH = PROJECT_ROOT / "data" / "cineanalytica.db"
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("SELECT review_text, sentiment FROM reviews", conn)

df.head(), df.shape

(                                         review_text sentiment
 0  One of the other reviewers has mentioned that ...  positive
 1  A wonderful little production. <br /><br />The...  positive
 2  I thought this was a wonderful way to spend ti...  positive
 3  Basically there's a family where a little boy ...  negative
 4  Petter Mattei's "Love in the Time of Money" is...  positive,
 (50000, 2))

In [2]:
df = df.dropna(subset=["review_text", "sentiment"]).copy()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()

df["label"] = df["sentiment"].map({"negative": 0, "positive": 1})
df["label"].value_counts(dropna=False)

label
1    25000
0    25000
Name: count, dtype: int64

In [3]:
### train baseline NLP model

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df["review_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=50000,
        ngram_range=(1,2)
    )),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
acc

0.9024

In [5]:
print("Accuracy:", acc)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

Accuracy: 0.9024
[[4459  541]
 [ 435 4565]]
              precision    recall  f1-score   support

           0      0.911     0.892     0.901      5000
           1      0.894     0.913     0.903      5000

    accuracy                          0.902     10000
   macro avg      0.903     0.902     0.902     10000
weighted avg      0.903     0.902     0.902     10000



In [6]:
import joblib
from pathlib import Path

MODEL_DIR = Path().resolve().parent / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(model, MODEL_DIR / "sentiment_model.joblib")

print("Saved:", MODEL_DIR / "sentiment_model.joblib")

Saved: /Users/khushdomadiya/Desktop/CineAnalytica/models/sentiment_model.joblib


In [7]:
### prediction helper

In [8]:
def predict_sentiment(text, model):
    pred = model.predict([text])[0]
    proba = model.predict_proba([text])[0]
    
    label = "Positive" if pred == 1 else "Negative"
    confidence = float(max(proba))
    
    return label, confidence

In [9]:
loaded_model = joblib.load(MODEL_DIR / "sentiment_model.joblib")

sample = "This movie was absolutely fantastic with great acting and story!"
predict_sentiment(sample, loaded_model)

('Positive', 0.9216137501058059)